# Historical analysis — public source copy
Outputs and execution counts were cleared for publication privacy. Original evidence remains local.
Do not Run All modeling notebooks, retrain frozen models or reevaluate the consumed test.
Dataset/model assets are withheld; see the release README.

# 04 — Leakage-aware preprocessing and Logistic Regression

**Phase 4 only.** Build a reproducible portfolio classifier for the existing ChurnFlag label. Fit on training customers and evaluate development candidates on validation customers. **The test partition remains reserved for Phase 5:** no test transformations, predictions, model metrics or threshold choice.

This snapshot has no fully documented feature cutoff or prediction horizon. Earlier EDA descriptively covered the whole dataset. Neither the split nor this baseline establishes prospective production performance.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "src/churn_analytics").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/churn_analytics").is_dir(), "Launch from project root or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
from churn_analytics.cleaning import DEFAULT_CLEAN_PATH, load_cleaned_dataset
from churn_analytics.model_splits import (create_split_manifest, development_split,
    validate_split_manifest, validate_target)
from churn_analytics.modeling import (FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    feature_policy, build_preprocessor, evaluate_pipeline)
from churn_analytics.model_reporting import metric_table
from churn_analytics.phase4 import run_phase4
REPORTS = ROOT / "reports/phase4"
MODELS = ROOT / "models"
print("Python executable:", sys.executable)

## 1. Business problem and target

ChurnFlag=1 corresponds to Churn=Yes; 0 corresponds to No. Churn is used only to validate that mapping. A possible use case is researching customer-outreach prioritization, but neither predictive accuracy nor an association demonstrates that outreach changes retention or saves money. No dated future churn horizon is claimed.

## 2. Dataset overview — schema and administrative input validation

Reuse the existing processed CSV. Do not recreate cleaning, overwrite source data, or perform new full-data feature EDA. Global shape/target/missing-charge checks below verify the expected input; they do not fit any model or summarize test-specific predictors.

In [ ]:
processed_sha = hashlib.sha256(DEFAULT_CLEAN_PATH.read_bytes()).hexdigest()
frame = load_cleaned_dataset()
target = validate_target(frame)
print("Validated processed shape:", frame.shape)
print("Target mapping counts:", target.value_counts().sort_index().to_dict())
print("Retained missing TotalCharges:", int(frame.TotalCharges.isna().sum()))
print("Dataset SHA-256:", processed_sha)

## 3. Feature eligibility and leakage risks

Only an explicit 18-column allowlist enters the model: two numeric measures and 16 categorical attributes. SeniorCitizen is categorical. IDs and both outcome representations are excluded. TotalCharges and its missingness flag are excluded because pre-outcome availability is not documented; fixed tenure/charge groups are redundant with included numeric values.

Even retained attributes are only plausibly available before churn. Confirmed feature timestamps and a prediction horizon would be needed for deployment. No additional feature selection is based on full-data SQL findings or final-test data.

In [ ]:
display(feature_policy())
print("Numerical:", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
assert len(FEATURES) == 18
assert not {"Churn", "ChurnFlag", "customerID", "TotalCharges", "TotalChargesMissing",
            "TenureGroup", "MonthlyChargeGroup"}.intersection(FEATURES)

## 4–5. Class imbalance and reproducible train/validation/test splitting

Sort customer IDs, stratify 80/20 into development/test, then stratify development 75/25 into train/validation, using seed 42 each time. All customers must be assigned exactly once. Split target counts below are administrative stratification metadata, not test-model evaluation. Only train/validation rows can be materialized by the modeling helper.

A majority-class classifier can be roughly 73% accurate while missing all churned customers. Model development therefore needs positive-class recall/precision and ranking metrics, not accuracy alone.

In [ ]:
manifest = create_split_manifest(frame, processed_sha)
validate_split_manifest(frame, manifest, processed_sha)
split_table = pd.DataFrame([{"split": role, "customers": info["size"],
    "non_churn": info["class_counts"]["0"], "churn": info["class_counts"]["1"],
    "churn_pct": 100*info["churn_proportion"]} for role, info in manifest["splits"].items()])
display(split_table)
ids = [identifier for info in manifest["splits"].values() for identifier in info["customer_ids"]]
assert len(ids) == len(set(ids)) == len(frame)
train = development_split(frame, manifest, "train")
validation = development_split(frame, manifest, "validation")
assert not set(train.X.index).intersection(validation.X.index)
del frame, target  # Full-data features are not retained for any model call.
print("The test set is represented by IDs/count metadata only; no test X/y is created.")

## 6. Numerical and categorical preprocessing

A ColumnTransformer combines numerical median imputation and StandardScaler with categorical constant-token imputation and OneHotEncoder. All learned medians, scales and observed category vocabularies come from training only. String conversion and explicit column selection do not estimate parameters.

The first training-observed category is dropped as a reference. Unseen categories encode as all zeros, which shares the reference representation. Missing values use __MISSING__; if that token was absent in training it is also unseen. This is robust to input shape/category changes but cannot learn an independent effect for an unseen group. Dropping references affects penalized estimates; structural correlations remain.

**Leakage example:** fitting a scaler on validation/test charges would make those customers influence training parameters. Fitting imputation or categories on the full DataFrame before splitting would have the same problem.

In [ ]:
preprocessor = build_preprocessor()  # Unfitted: no fit/transform is called here.
display(preprocessor)
assert not hasattr(preprocessor, "transformers_")
print("Predeclared selection: highest validation average precision among the two LR candidates; ties prefer standard.")
print("All reported classifications use probability >= 0.50; no threshold search.")

## 7. Dummy reference and reproducible workflow

The shared workflow persists configuration and split identities before fitting, trains DummyClassifier first, then standard and balanced Logistic Regression on training only. The dummy uses most_frequent rather than a learned customer ranking. The fitted candidate is saved without any train+validation or whole-dataset refit. Re-running preserves split identities and rejects changed source fingerprints.

In [ ]:
run_result = run_phase4()
metrics = json.loads((REPORTS / "metrics.json").read_text())
config = json.loads((REPORTS / "run_config.json").read_text())
saved_manifest = json.loads((REPORTS / "split_manifest.json").read_text())
assert manifest == saved_manifest
assert run_result["test_evaluated"] is False
comparison = metric_table(metrics)
display(comparison.loc[comparison.model.eq("dummy_most_frequent")])
dummy = metrics["models"]["dummy_most_frequent"]["validation"]
display(Markdown(f"**Actual validation baseline:** accuracy {dummy['accuracy']:.4f}, "
    f"churn recall {dummy['recall']:.4f}, average precision {dummy['average_precision']:.4f}. "
    "A majority-only label can be accurate yet identify no positive cases."))

## 8. Logistic Regression, regularization and convergence

The sigmoid maps a linear log-odds score to a probability: p=1/(1+exp(−z)). Logistic Regression is a classifier because that probability is compared with a classification threshold. L2 regularization shrinks coefficients; C=1 is the fixed inverse penalty strength. Both models use lbfgs, max_iter=2000, tol=1e-6 and seed 42. Deprecated penalty/n_jobs arguments are omitted for the installed sklearn API. No hyperparameter search is performed. Convergence warnings cause a failure instead of being silently ignored.

In [ ]:
display(comparison)
for name, values in metrics["models"].items():
    print(name, values["fit"])
print("Selected:", metrics["selected_model"])
print("Selection criterion:", config["selection_criterion"])
print("Effective configuration:", config["classifier"])
print("Training diagnostics:")
display(metric_table(metrics, "train"))

## 9. Confusion matrix and threshold-0.50 metric definitions

- TP: churned customer flagged as churn.
- TN: No-labeled customer predicted No.
- FP: No-labeled customer flagged as churn.
- FN: churned customer missed by the flag.

Accuracy=(TP+TN)/n; precision=TP/(TP+FP); recall=TP/(TP+FN); F1 is their harmonic mean. Zero-division results are set to zero for positive-class precision/recall/F1. False negatives can miss outreach opportunities; false positives can produce unnecessary contacts or costs. Neither is assigned an invented monetary value.

In [ ]:
def show_chart(name, width=1100):
    display(Image(filename=str(REPORTS / "figures" / f"{name}.png"), width=width))
selected = metrics["selected_model"]
selected_metrics = metrics["models"][selected]["validation"]
show_chart("validation_confusion_matrices", width=1400)
tn, fp = selected_metrics["confusion_matrix"][0]
fn, tp = selected_metrics["confusion_matrix"][1]
display(Markdown(f"**Actual selected-model counts:** TN={tn}, FP={fp}, FN={fn}, TP={tp}. "
    f"Precision={selected_metrics['precision']:.4f}; recall={selected_metrics['recall']:.4f}; "
    f"F1={selected_metrics['f1']:.4f}. These are validation outcomes only."))

## 10. ROC, precision–recall and probability distributions

ROC-AUC summarizes ranking of positives relative to negatives across thresholds. Average precision summarizes positive-class precision weighted by recall increments; it is not the trapezoidal PR area. Validation prevalence provides a reference for AP. A probability in [0,1] is not proof of calibration or future customer risk.

In [ ]:
show_chart("validation_roc_curves", width=900)
show_chart("validation_precision_recall_curves", width=900)
show_chart("validation_probability_distributions", width=1400)
all_predictions = pd.read_csv(REPORTS / "candidate_validation_predictions.csv")
assert set(all_predictions.split) == {"validation"}
assert all_predictions.churn_probability.between(0, 1).all()
assert not set(all_predictions.customerID).intersection(manifest["splits"]["test"]["customer_ids"])
for name, values in metrics["models"].items():
    print(name, "validation probability summary:", values["validation"]["probability_summary"])

## 11. Class weighting and candidate selection

Balanced weights are learned from training label counts via n/(2×class_count). They change the training objective, often moving the precision/recall trade-off at the same threshold. They do not guarantee better ranking, calibrated probabilities, business savings or an optimal outreach decision.

The fixed selection criterion is validation average precision. The model with the highest F1 or recall need not win that criterion. A small AP difference on one validation split is not a statistically established performance advantage.

In [ ]:
state = json.loads((REPORTS / "preprocessing_state.json").read_text())
print("Balanced training weights:", state["class_weight_balanced_training_weights"])
display(comparison.loc[comparison.model.str.startswith("logistic")])
standard = metrics["models"]["logistic_standard"]["validation"]
balanced = metrics["models"]["logistic_balanced"]["validation"]
display(Markdown(f"**Observed trade-off:** standard recall/precision "
    f"{standard['recall']:.4f}/{standard['precision']:.4f}; balanced "
    f"{balanced['recall']:.4f}/{balanced['precision']:.4f}. "
    f"AP is {standard['average_precision']:.4f} versus {balanced['average_precision']:.4f}. "
    "No operationally optimal threshold or causal retention effect follows from these comparisons."))

## 12. Coefficients, references and conditional odds ratios

Transformed feature names must align with the fitted coefficient order. exp(β) is a conditional odds multiplier, not a probability ratio. Numerical coefficients are per training standard deviation, not per raw month/charge unit. Categorical coefficients compare with saved references. L2 regularization and correlated/structurally dependent service attributes complicate interpretation. Coefficients are fitted associations, not causal feature importance.

In [ ]:
coefficients = pd.read_csv(REPORTS / "coefficients.csv")
coefficient_metadata = json.loads((REPORTS / "coefficient_metadata.json").read_text())
print("Intercept:", coefficient_metadata["intercept"])
print("Reference categories:", coefficient_metadata["reference_categories"])
print("Numeric training scales:", coefficient_metadata["numeric_training_scales"])
display(coefficients.loc[coefficients.source_feature.isin(NUMERIC_FEATURES)])
display(coefficients.nlargest(10, "coefficient"))
display(coefficients.nsmallest(10, "coefficient"))
show_chart("logistic_coefficient_associations", width=1100)

### Trusted serialization and validation-only reload verification

The serialized artifact contains preprocessing and classifier together, already fitted on training customers. Loading pickle/joblib can execute code: never load an untrusted file or assume portability across versions. The file below was just created by this notebook's workflow, and its fingerprint is checked before its validation-only reload.

In [ ]:
import joblib
artifact = MODELS / "logistic_regression_pipeline.joblib"
audit = json.loads((REPORTS / "model_audit.json").read_text())
assert hashlib.sha256(artifact.read_bytes()).hexdigest() == audit["model_sha256"]
owned_pipeline = joblib.load(artifact)  # Only the artifact this run just created.
_, reload_predictions = evaluate_pipeline(owned_pipeline, validation, manifest)
expected = pd.read_csv(REPORTS / "validation_predictions.csv")
np.testing.assert_allclose(reload_predictions.churn_probability, expected.churn_probability,
                           rtol=1e-12, atol=1e-12)
assert coefficients.transformed_feature.tolist() == owned_pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()
print("Training-fitted Pipeline reload reproduces validation probabilities; no refit.")

## 13–14. Business implications, assumptions and limitations

Use the results to discuss positive-class detection, ranking and outreach trade-offs. There is no actual outreach experiment, cost matrix, prediction horizon, documented feature cutoff, prospective validation or retention ROI. Balanced probabilities may need calibration research; no calibration or threshold optimization is performed here.

Prior full-data EDA means the future test partition has already been seen descriptively. Keep it out of further model development, report that limitation, and seek new timestamped data for genuinely independent confirmation. A single validation split and two-candidate selection can overfit development evidence. Demographic suitability and subgroup behavior need assessment before any operational use.

Read the [Phase 4 report](../reports/PHASE4_LOGISTIC_REGRESSION_REPORT.md), [guide with interview answers](../docs/PHASE4_MODELING_GUIDE.md), and [verification record](../PHASE4_VERIFICATION.md). No Random Forest, test evaluation or Streamlit work is authorized in this phase.

Method references: [leakage prevention](https://scikit-learn.org/stable/common_pitfalls.html), [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).

In [ ]:
assert hashlib.sha256(DEFAULT_CLEAN_PATH.read_bytes()).hexdigest() == processed_sha
assert saved_manifest["test_evaluated"] is False
assert audit["test_evaluated"] is False
assert audit["test_predictions"] == audit["test_feature_rows_transformed"] == 0
assert all(event["split"] in {"train", "validation"} for event in audit["events"])
assert all(event["split"] == "train" for event in audit["events"] if event["operation"] == "fit_preprocessing_and_classifier")
assert state["scaler_samples_seen"] == manifest["splits"]["train"]["size"]
print("Phase 4 notebook checks passed. Test remains reserved; source CSV unchanged.")